# Seq2Seq + Bahdanau Attention — Train Attention Model Only

The normal Seq2Seq model is **already trained** in `seq2seq_basic.pkl`.

This notebook does **NOT** retrain the normal Seq2Seq from scratch.

Instead:

1. Load the trained basic Seq2Seq checkpoint.
2. Reuse its learned **embeddings, encoder LSTMs, decoder LSTMs and output layer**.
3. Add a new **Bahdanau attention** layer.
4. Train the attention-augmented model on the supplied training data.
5. Validate during training.
6. Evaluate with BLEU.
7. Compare the pretrained basic model and attention model.
8. Compare greedy and beam-search decoding.

The important distinction is:

**Basic model:** already trained → reuse its weights.

**Attention model:** new attention parameters → train the attention model.

In [25]:
# 1. Imports

import pickle
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [44]:
# 2. Configuration

DATA_DIR = Path(".")

PKL_PATH = DATA_DIR / "seq2seq_basic.pkl"
TRAIN_PATH = DATA_DIR / "en_fr_train.csv"
VAL_PATH = DATA_DIR / "en_fr_val.csv"
TEST_PATH = DATA_DIR / "en_fr_test.csv"

# Training configuration
BATCH_SIZE = 256
EPOCHS = 3
LEARNING_RATE = 1e-3
CLIP_NORM = 1.0
PATIENCE = 2

# Set to an integer for a quick test run.
# None = use the complete training/validation data.
TRAIN_LIMIT = None
VAL_LIMIT = None

# Number of test examples for BLEU.
# None = all test examples.
TEST_LIMIT = 1000

for path in [PKL_PATH, TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

print("All files found.")

All files found.


In [27]:
# 3. Load the already-trained basic Seq2Seq checkpoint

with open(PKL_PATH, "rb") as f:
    checkpoint = pickle.load(f)

cfg = checkpoint["cfg"]
basic_weights = checkpoint["model"]

src_itos = checkpoint["src_itos"]
tgt_itos = checkpoint["tgt_itos"]

src_stoi = {w: i for i, w in enumerate(src_itos)}
tgt_stoi = {w: i for i, w in enumerate(tgt_itos)}

PAD, SOS, EOS, UNK = 0, 1, 2, 3

MAX_LEN = cfg["max_len"]
MAX_DECODE_LEN = cfg["max_decode_len"]
EMB_DIM = cfg["emb_dim"]
HID_DIM = cfg["hid_dim"]
N_LAYERS = cfg["n_layers"]
DROPOUT = cfg["dropout"]

print("Checkpoint loaded.")
print("Vocabulary:", len(src_itos))
print("Embedding:", EMB_DIM)
print("Hidden:", HID_DIM)
print("Layers:", N_LAYERS)
print("Max length:", MAX_LEN)
print("Saved arrays:", len(basic_weights))

Checkpoint loaded.
Vocabulary: 16000
Embedding: 256
Hidden: 512
Layers: 2
Max length: 40
Saved arrays: 16


In [28]:
# 4. Load datasets

train_df = pd.read_csv(TRAIN_PATH).dropna()
val_df = pd.read_csv(VAL_PATH).dropna()
test_df = pd.read_csv(TEST_PATH).dropna()

train_df = train_df[["en", "fr"]].astype(str)
val_df = val_df[["en", "fr"]].astype(str)
test_df = test_df[["en", "fr"]].astype(str)

if TRAIN_LIMIT is not None:
    train_df = train_df.iloc[:TRAIN_LIMIT]

if VAL_LIMIT is not None:
    val_df = val_df.iloc[:VAL_LIMIT]

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 232825
Validation: 890
Test: 8597


In [29]:
# 5. Tokenization

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())


def encode_source(text):
    ids = [src_stoi.get(t, UNK) for t in tokenize(text)]
    ids = ids[:MAX_LEN - 1]
    ids.append(EOS)
    return ids


def encode_target(text):
    ids = [tgt_stoi.get(t, UNK) for t in tokenize(text)]
    ids = ids[:MAX_LEN - 2]
    return [SOS] + ids + [EOS]


def make_arrays(df):
    src = [encode_source(x) for x in df["en"]]
    tgt = [encode_target(x) for x in df["fr"]]

    src = tf.keras.utils.pad_sequences(
        src, maxlen=MAX_LEN, padding="post", value=PAD
    )

    tgt = tf.keras.utils.pad_sequences(
        tgt, maxlen=MAX_LEN, padding="post", value=PAD
    )

    return src.astype("int32"), tgt.astype("int32")


print("Preparing training data...")
X_train, Y_train = make_arrays(train_df)

print("Preparing validation data...")
X_val, Y_val = make_arrays(val_df)

print("Train tensors:", X_train.shape, Y_train.shape)
print("Validation tensors:", X_val.shape, Y_val.shape)

Preparing training data...
Preparing validation data...
Train tensors: (232825, 40) (232825, 40)
Validation tensors: (890, 40) (890, 40)


## 6. Attention architecture

We use **Bahdanau/additive attention**.

At every decoder step:

```text
decoder hidden state
        ↓
   attention
        ↓
weighted encoder states
        ↓
context vector
        ↓
decoder
        ↓
next French word
```

The basic model used only the final encoder state.

The attention model can look at **all encoder states**.

In [30]:
# 6. Encoder

class Encoder(keras.Model):
    def __init__(self):
        super().__init__()

        self.embedding = layers.Embedding(
            len(src_itos),
            EMB_DIM
        )

        self.rnns = [
            layers.LSTM(
                HID_DIM,
                return_sequences=True,
                return_state=True
            )
            for _ in range(N_LAYERS)
        ]

        self.dropout = layers.Dropout(DROPOUT)

    def call(self, src, training=False):
        mask = tf.not_equal(src, PAD)

        x = self.embedding(src)
        x = self.dropout(x, training=training)

        hs, cs = [], []

        for rnn in self.rnns:
            x, h, c = rnn(
                x,
                mask=mask,
                training=training
            )
            hs.append(h)
            cs.append(c)

        return x, tf.stack(hs), tf.stack(cs), mask

In [31]:
# 7. Bahdanau attention

class BahdanauAttention(layers.Layer):
    def __init__(self, hidden_size):
        super().__init__()

        self.W1 = layers.Dense(hidden_size, use_bias=False)
        self.W2 = layers.Dense(hidden_size, use_bias=False)
        self.V = layers.Dense(1, use_bias=False)

    def call(self, query, values, mask):
        # query:  [batch, hidden]
        # values: [batch, source_len, hidden]

        query = tf.expand_dims(query, 1)

        score = self.V(
            tf.nn.tanh(
                self.W1(values) + self.W2(query)
            )
        )

        score = tf.squeeze(score, axis=-1)

        # Ignore PAD positions in the source.
        score = tf.where(
            mask,
            score,
            tf.fill(tf.shape(score), tf.constant(-1e9))
        )

        weights = tf.nn.softmax(score, axis=1)

        context = tf.reduce_sum(
            values * tf.expand_dims(weights, -1),
            axis=1
        )

        return context, weights

In [32]:
# 8. Attention Decoder

class AttentionDecoder(keras.Model):
    def __init__(self):
        super().__init__()

        self.embedding = layers.Embedding(
            len(tgt_itos),
            EMB_DIM
        )

        self.attention = BahdanauAttention(HID_DIM)

        # Attention gives:
        # embedding = 256
        # context   = 512
        # total     = 768
        #
        # Pretrained decoder expects 256 features.
        self.context_projection = layers.Dense(
            EMB_DIM,
            activation="tanh"
        )

        self.rnns = [
            layers.LSTM(
                HID_DIM,
                return_sequences=False,
                return_state=True
            )
            for _ in range(N_LAYERS)
        ]

        self.dropout = layers.Dropout(DROPOUT)

        self.fc = layers.Dense(len(tgt_itos))

    def step(
        self,
        token,
        state_h,
        state_c,
        encoder_outputs,
        src_mask,
        training=False
    ):

        # -------------------------------------------------
        # token shape:
        # (batch, 1)
        # -------------------------------------------------

        x = self.embedding(token)

        # Force:
        # (batch, 1, EMB_DIM)
        x = tf.reshape(
            x,
            [tf.shape(token)[0], 1, EMB_DIM]
        )

        x = self.dropout(
            x,
            training=training
        )

        # -------------------------------------------------
        # Bahdanau attention
        # -------------------------------------------------

        # Last decoder hidden state:
        # (batch, HID_DIM)
        query = state_h[-1]

        context, attention_weights = self.attention(
            query,
            encoder_outputs,
            src_mask
        )

        # context:
        # (batch, HID_DIM)
        #
        # Convert to:
        # (batch, 1, HID_DIM)
        context = tf.expand_dims(
            context,
            axis=1
        )

        # -------------------------------------------------
        # Combine decoder embedding + attention context
        # -------------------------------------------------

        # x:
        # (batch, 1, 256)
        #
        # context:
        # (batch, 1, 512)
        #
        # result:
        # (batch, 1, 768)

        x = tf.concat(
            [x, context],
            axis=-1
        )

        # -------------------------------------------------
        # 768 -> 256
        # -------------------------------------------------

        x = self.context_projection(x)

        # Explicitly guarantee:
        # (batch, 1, 256)
        x = tf.reshape(
            x,
            [tf.shape(token)[0], 1, EMB_DIM]
        )

        # -------------------------------------------------
        # Pretrained LSTM layers
        # -------------------------------------------------

        new_h = []
        new_c = []

        for i, rnn in enumerate(self.rnns):

            x, h, c = rnn(
                x,
                initial_state=[
                    state_h[i],
                    state_c[i]
                ],
                training=training
            )

            new_h.append(h)
            new_c.append(c)

            # The LSTM output is:
            # (batch, 512)
            #
            # For the next LSTM we need:
            # (batch, 1, 512)
            #
            # So expand it back to a sequence of length 1.
            if i < len(self.rnns) - 1:
                x = tf.expand_dims(
                    x,
                    axis=1
                )

        # -------------------------------------------------
        # Output layer
        # -------------------------------------------------

        logits = self.fc(
            self.dropout(
                x,
                training=training
            )
        )

        # logits:
        # (batch, 1, target_vocab)

        return (
            logits,
            tf.stack(new_h),
            tf.stack(new_c),
            attention_weights
        )

In [33]:
# 9. Create model and initialize from pretrained Seq2Seq

encoder = Encoder()
decoder = AttentionDecoder()

# ---------------------------------------------------------
# Build encoder
# ---------------------------------------------------------

dummy_src = tf.zeros(
    (1, MAX_LEN),
    dtype=tf.int32
)

enc_out, h, c, src_mask = encoder(
    dummy_src,
    training=False
)

print("Encoder output:", enc_out.shape)
print("Encoder h:", h.shape)
print("Encoder c:", c.shape)

# ---------------------------------------------------------
# Build decoder
# ---------------------------------------------------------

dummy_token = tf.constant(
    [[SOS]],
    dtype=tf.int32
)

logits, new_h, new_c, attn = decoder.step(
    dummy_token,
    h,
    c,
    enc_out,
    src_mask,
    training=False
)

print("Decoder output:", logits.shape)

print("Model architecture built successfully.")

Encoder output: (1, 40, 512)
Encoder h: (2, 1, 512)
Encoder c: (2, 1, 512)
Decoder output: (1, 16000)
Model architecture built successfully.


In [34]:
# 10. Loss and optimizer

loss_object = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    clipnorm=CLIP_NORM
)


def masked_loss(y_true, logits):
    loss = loss_object(y_true, logits)

    mask = tf.cast(
        tf.not_equal(y_true, PAD),
        loss.dtype
    )

    return tf.reduce_sum(loss * mask) / tf.reduce_sum(mask)

In [35]:
# 11. One training step

@tf.function
def train_step(src, tgt):
    total_loss = 0.0
    steps = tf.shape(tgt)[1] - 1

    with tf.GradientTape() as tape:

        enc_out, h, c, src_mask = encoder(
            src,
            training=True
        )

        state_h = h
        state_c = c

        # Teacher forcing:
        # input  = <sos>, previous French word
        # target = next French word
        for t in tf.range(steps):

            token = tgt[:, t:t+1]
            target = tgt[:, t+1]

            logits, state_h, state_c, _ = decoder.step(
                token,
                state_h,
                state_c,
                enc_out,
                src_mask,
                training=True
            )

            total_loss += masked_loss(
                target,
                logits
            )

        total_loss /= tf.cast(steps, tf.float32)

    variables = (
        encoder.trainable_variables +
        decoder.trainable_variables
    )

    gradients = tape.gradient(
        total_loss,
        variables
    )

    optimizer.apply_gradients(
        zip(gradients, variables)
    )

    return total_loss

In [36]:
# 12. Validation loss

@tf.function
def validation_step(src, tgt):
    total_loss = 0.0
    steps = tf.shape(tgt)[1] - 1

    enc_out, h, c, src_mask = encoder(
        src,
        training=False
    )

    state_h = h
    state_c = c

    for t in tf.range(steps):

        token = tgt[:, t:t+1]
        target = tgt[:, t+1]

        logits, state_h, state_c, _ = decoder.step(
            token,
            state_h,
            state_c,
            enc_out,
            src_mask,
            training=False
        )

        total_loss += masked_loss(
            target,
            logits
        )

    return total_loss / tf.cast(steps, tf.float32)

In [37]:
TRAIN_LIMIT = 10000
VAL_LIMIT = 1000

In [38]:
X_train_small = X_train[:TRAIN_LIMIT]
Y_train_small = Y_train[:TRAIN_LIMIT]

X_val_small = X_val[:VAL_LIMIT]
Y_val_small = Y_val[:VAL_LIMIT]

In [39]:
# 13. tf.data datasets

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train, Y_train)
).shuffle(
    min(len(X_train), 50000)
).batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (X_val, Y_val)
).batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

print("Datasets ready.")

Datasets ready.


In [45]:
# 14. Train ONLY the attention model

best_val = float("inf")
bad_epochs = 0

history = []

for epoch in range(1, EPOCHS + 1):

    start = time.time()

    train_losses = []

    for src_batch, tgt_batch in train_ds:
        loss = train_step(
            src_batch,
            tgt_batch
        )
        train_losses.append(float(loss))

    val_losses = []

    for src_batch, tgt_batch in val_ds:
        loss = validation_step(
            src_batch,
            tgt_batch
        )
        val_losses.append(float(loss))

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))

    elapsed = time.time() - start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "time_sec": elapsed
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {elapsed/60:.1f} min"
    )

    if val_loss < best_val:
      best_val = val_loss
      bad_epochs = 0

      # Save encoder weights
      encoder.save_weights(
          "attention_encoder.weights.h5"
      )

      # Save decoder weights manually
      decoder_weights = decoder.get_weights()

      with open(
          "attention_decoder.weights.pkl",
          "wb"
      ) as f:
          pickle.dump(
              decoder_weights,
              f
          )

      print("  saved best attention model")

    else:
        bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

history_df = pd.DataFrame(history)
display(history_df)

Epoch 1/3 | Train Loss: 5.4405 | Val Loss: 4.5977 | Time: 14.4 min
  saved best attention model
Epoch 2/3 | Train Loss: 4.1518 | Val Loss: 3.8683 | Time: 14.5 min
  saved best attention model
Epoch 3/3 | Train Loss: 3.5868 | Val Loss: 3.4983 | Time: 14.5 min
  saved best attention model


,epoch,train_loss,val_loss,time_sec
0,1,5.440533,4.597731,862.556085
1,2,4.151803,3.868347,871.138598
2,3,3.586801,3.498321,871.995478


In [50]:
# 15. Load the best attention model

import pickle

# Load encoder
encoder.load_weights("attention_encoder.weights.h5")

# Load decoder weights from PKL
with open("attention_decoder.weights.pkl", "rb") as f:
    decoder_weights = pickle.load(f)

decoder.set_weights(decoder_weights)

print("Best attention checkpoint loaded.")

Best attention checkpoint loaded.


In [63]:
# Create source vocabulary mappings
src_stoi = {word: i for i, word in enumerate(src_itos)}
tgt_stoi = {word: i for i, word in enumerate(tgt_itos)}

# Source special tokens
PAD_SRC = src_stoi.get("<pad>", 0)
UNK_SRC = src_stoi.get("<unk>", 3)

# Target special tokens
SOS = tgt_stoi.get("<sos>", 1)
EOS = tgt_stoi.get("<eos>", 2)

print("PAD_SRC:", PAD_SRC)
print("UNK_SRC:", UNK_SRC)
print("SOS:", SOS)
print("EOS:", EOS)

PAD_SRC: 0
UNK_SRC: 3
SOS: 1
EOS: 2


In [64]:
def encode_input(text):
    words = text.lower().strip().split()

    tokens = [
        src_stoi.get(word, UNK_SRC)
        for word in words
    ]

    tokens = [SOS] + tokens + [EOS]

    tokens = tokens[:MAX_LEN]

    tokens += [PAD_SRC] * (MAX_LEN - len(tokens))

    src = tf.constant([tokens], dtype=tf.int32)

    return src, tokens

In [65]:
# 16. Greedy decoding with attention

def attention_greedy_decode(text, return_attention=False):

    src, _ = encode_input(text)

    enc_out, h, c, src_mask = encoder(
        src,
        training=False
    )

    token = SOS
    output = []
    all_attention = []

    for _ in range(MAX_DECODE_LEN):

        token_tensor = tf.constant(
            [[token]],
            dtype=tf.int32
        )

        logits, h, c, attn = decoder.step(
            token_tensor,
            h,
            c,
            enc_out,
            src_mask,
            training=False
        )

        token = int(
            tf.argmax(logits[0]).numpy()
        )

        if return_attention:
            all_attention.append(
                attn[0].numpy()
            )

        if token == EOS:
            break

        output.append(token)

    words = [
        tgt_itos[i]
        for i in output
    ]

    if return_attention:
        return words, np.array(all_attention)

    return words


example = test_df.iloc[0]["en"]

print("EN :", example)
print("REF:", test_df.iloc[0]["fr"])
print("ATT:", " ".join(
    attention_greedy_decode(example)
))

EN : Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.
REF: Il y a plusieurs années, ici à Ted, Peter Skillman a présenté une épreuve de conception appelée l'épreuve du marshmallow.
ATT: il y a quelques années , il y a un <unk> <unk> <unk> <unk> <unk> à l ' <unk> <unk> <unk>


In [66]:
# 17. BLEU evaluation

try:
    import sacrebleu
    HAVE_SACREBLEU = True
except ImportError:
    HAVE_SACREBLEU = False


def calculate_bleu(predictions, references):

    predictions = [
        " ".join(p)
        for p in predictions
    ]

    references = [
        " ".join(r)
        for r in references
    ]

    if HAVE_SACREBLEU:
        return sacrebleu.corpus_bleu(
            predictions,
            [references],
            tokenize="none"
        ).score

    from nltk.translate.bleu_score import corpus_bleu

    return 100 * corpus_bleu(
        [[r] for r in references],
        [p.split() for p in predictions]
    )

In [67]:
# 18. Evaluate the attention model

if TEST_LIMIT is None:
    eval_df = test_df
else:
    eval_df = test_df.iloc[:TEST_LIMIT]

references = [
    tokenize(x)
    for x in eval_df["fr"]
]

start = time.time()

attention_predictions = [
    attention_greedy_decode(x)
    for x in eval_df["en"]
]

attention_time = time.time() - start

attention_bleu = calculate_bleu(
    attention_predictions,
    references
)

print("Attention BLEU:", round(attention_bleu, 2))
print("Evaluation time:", round(attention_time / 60, 2), "minutes")

Attention BLEU: 0.0
Evaluation time: 8.86 minutes


/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

In [68]:
# 19. Compare a few predictions

for i in range(min(5, len(eval_df))):

    print("=" * 100)

    print("EN :")
    print(eval_df.iloc[i]["en"])

    print("\nREFERENCE:")
    print(eval_df.iloc[i]["fr"])

    print("\nATTENTION:")
    print(" ".join(attention_predictions[i]))

EN :
Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.

REFERENCE:
Il y a plusieurs années, ici à Ted, Peter Skillman a présenté une épreuve de conception appelée l'épreuve du marshmallow.

ATTENTION:
il y a quelques années , il y a un <unk> <unk> <unk> <unk> <unk> à l ' <unk> <unk> <unk>
EN :
And the idea's pretty simple:  Teams of four have to build the tallest free-standing structure  out of 20 sticks of spaghetti,  one yard of tape, one yard of string  and a marshmallow.

REFERENCE:
Et l'idée est plutôt simple. Des équipes de quatre personnes doivent bâtir la plus haute structure tenant debout avec 20 spaghettis, un mètre de ruban collant, un mètre de ficelle, et un marshmallow.

ATTENTION:
<unk> , les <unk> <unk> <unk> des <unk> de <unk> de <unk> <unk> de <unk> <unk> de <unk> de <unk> de <unk> , et un <unk> de <unk> , un <unk> de
EN :
The marshmallow has to be on top.

REFERENCE:
Le marshmallow doit être placé au sommet

In [69]:
# 20. Save the trained attention checkpoint

attention_checkpoint = {
    "cfg": cfg,
    "src_itos": src_itos,
    "tgt_itos": tgt_itos,
    "encoder_weights": encoder.get_weights(),
    "decoder_weights": decoder.get_weights()
}

with open(
    "seq2seq_attention.pkl",
    "wb"
) as f:
    pickle.dump(
        attention_checkpoint,
        f
    )

print("Saved: seq2seq_attention.pkl")

Saved: seq2seq_attention.pkl


## What you have after running this notebook

You will have:

- `seq2seq_basic.pkl` → your original trained normal Seq2Seq
- `seq2seq_attention.pkl` → the newly trained attention model
- `attention_encoder.weights.h5`
- `attention_decoder.weights.h5`

The attention model starts from the **already learned basic Seq2Seq weights**, then learns the new Bahdanau attention parameters and fine-tunes the complete model.

### Assignment comparison

| Model | Training |
|---|---|
| Basic Seq2Seq | Already trained — loaded from `.pkl` |
| Seq2Seq + Attention | Trained in this notebook |

The BLEU comparison should be made using the **same test set** and the same preprocessing.